In [102]:
import pandas as pd

df = pd.read_parquet('./files/results/data_finished.parquet', engine='pyarrow')
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 48447 entries, 0 to 48446
Data columns (total 71 columns):
 #   Column                                Non-Null Count  Dtype  
---  ------                                --------------  -----  
 0   id_discente                           48447 non-null  int64  
 1   ano                                   48447 non-null  int64  
 2   periodo                               48447 non-null  int64  
 3   situacao                              48447 non-null  str    
 4   sexo                                  48447 non-null  str    
 5   estado_civil                          48447 non-null  str    
 6   raca_declarada                        48447 non-null  str    
 7   ano_ingresso                          48447 non-null  float64
 8   periodo_ingresso                      48447 non-null  float64
 9   status_discente                       48447 non-null  str    
 10  forma_ingresso                        48447 non-null  str    
 11  quantidade_membros_familia

In [103]:
target_cols = {
    'target_percntil_media': ['media_geral', 'status_discente', 'situacao'],
    'target_reprov': ['situacao', 'status_discente'],
    'target_trancamento': ['situacao', 'status_discente', 'faixa_renda_familiar'],
    'target_risco': ['media_geral', 'situacao', 'status_discente', 'ch_integralizada', 'ch_pendente']
}

df = df.drop([f"{col}_missing" for col, _ in target_cols.items()], axis=1)
df["semestre"] = df["ano_ingresso"] * 10 + df["periodo_ingresso"]

In [104]:
# =========================================================
#  DIVISÃO TREINO / TESTE TEMPORAL
# =========================================================
CUTOFF = 20222

ANO_ATUAL = df.ano_ingresso.max()
data = df[df["ano_ingresso"] <= ANO_ATUAL - 1]

train = data[data["semestre"] <= CUTOFF].copy()
test  = data[data["semestre"] > CUTOFF].copy()

print("proporção treino:", len(train) / len(data))
print("proporção teste :", len(test) / len(data))


print('-='*50)
for target_col, _ in target_cols.items():
    print(f"{target_col} treino:", train[target_col].mean())
    print(f"{target_col} teste :", test[target_col].mean())
    print('-='*50)


proporção treino: 0.6172736425922742
proporção teste : 0.38272635740772576
-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=
target_percntil_media treino: 0.1479551241247613
target_percntil_media teste : 0.023163298042990055
-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=
target_reprov treino: 0.0753898790579249
target_reprov teste : 0.009432146294513956
-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=
target_trancamento treino: 0.027211966900063653
target_trancamento teste : 0.003400705806865576
-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=
target_risco treino: 0.04507479312539783
target_risco teste : 0.004683991017003529
-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=


In [ ]:
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    f1_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier


def apply_model(col, deps):
    df_model = df.copy()

    # ordenar tempo
    df_model["semestre"] = df_model["ano"]*10 + df_model["periodo"]
    df_model = df_model.sort_values(["id_discente", "semestre"])

    # criar target futuro
    df_model[f"{col}_t1"] = df_model.groupby("id_discente")[col].shift(-1)

    # remover sem futuro
    df_model = df_model.dropna(subset=[f"{col}_t1"])

    # remover outros targets e suas dependencias
    df_model = df_model.drop([c for c in target_cols if c != col] + deps + ['id_discente'], axis=1)

    # encoding
    df_cat = df_model.select_dtypes(include=["object","str"]).columns
    df_model = pd.get_dummies(df_model, columns=df_cat, drop_first=True, dtype=int)

    # split temporal
    train = df_model[df_model["semestre"] <= CUTOFF]
    test  = df_model[df_model["semestre"] > CUTOFF]

    # X e y alinhados
    X_train = train.drop(columns=[col, f"{col}_t1"])
    y_train = train[f"{col}_t1"]

    X_test = test.drop(columns=[col, f"{col}_t1"])
    y_test = test[f"{col}_t1"]

    # CV temporal
    cv = TimeSeriesSplit(n_splits=5)

    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            solver="saga",
            max_iter=1000,
            class_weight="balanced",
            random_state=1
        ))
    ])

    param_grid = {
        "model__C": [0.01, 0.1, 1, 10],
        "model__penalty": ["l1", "l2"]
    }

    grid = GridSearchCV(pipe, param_grid=param_grid, cv=cv)
    grid.fit(X_train, y_train)

    return {
        "Target": col,
        "Best Params": grid.best_params_,
        "Score": grid.best_score_,
        "Best Estimator": grid.best_estimator_,
        "X_train": X_train,
        "Y_train": y_train,
        "X_test": X_test,
        "y_test": y_test
    }


In [107]:
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import pandas as pd


def apply_model(col, deps):
    df_model = df.copy()

    # =============================
    # PREPARAÇÃO
    # =============================
    df_model["semestre"] = df_model["ano"]*10 + df_model["periodo"]
    df_model = df_model.sort_values(["id_discente", "semestre"])

    df_model[f"{col}_t1"] = df_model.groupby("id_discente")[col].shift(-1)
    df_model = df_model.dropna(subset=[f"{col}_t1"])

    df_model = df_model.drop(
        [c for c in target_cols if c != col] + deps + ['id_discente'],
        axis=1
    )

    # encoding
    df_cat = df_model.select_dtypes(include=["object","str"]).columns
    df_model = pd.get_dummies(df_model, columns=df_cat, drop_first=True, dtype=int)

    # =============================
    # SPLIT TEMPORAL
    # =============================
    train = df_model[df_model["semestre"] <= CUTOFF]
    test  = df_model[df_model["semestre"] > CUTOFF]

    X_train = train.drop(columns=[col, f"{col}_t1"])
    y_train = train[f"{col}_t1"]

    X_test = test.drop(columns=[col, f"{col}_t1"])
    y_test = test[f"{col}_t1"]

    # =============================
    # CV
    # =============================
    cv = TimeSeriesSplit(n_splits=5)

    models = {}

    # =============================
    # 1. LOGISTIC REGRESSION
    # =============================
    pipe_logit = Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            solver="saga",
            max_iter=2000,
            class_weight="balanced",
            random_state=1
        ))
    ])

    param_logit = {
        "model__C": [0.01, 0.1, 1, 10],
        "model__penalty": ["l1", "l2"]
    }

    grid_logit = GridSearchCV(pipe_logit, param_logit, cv=cv, scoring="roc_auc")
    grid_logit.fit(X_train, y_train)

    models["Logistic"] = grid_logit

    # =============================
    # 2. RANDOM FOREST
    # =============================
    rf = RandomForestClassifier(random_state=1)

    param_rf = {
        "n_estimators": [100, 200],
        "max_depth": [5, 10, None],
        "min_samples_split": [2, 5]
    }

    grid_rf = GridSearchCV(rf, param_rf, cv=cv, scoring="roc_auc", n_jobs=-1)
    grid_rf.fit(X_train, y_train)

    models["RandomForest"] = grid_rf

    # =============================
    # 3. XGBOOST
    # =============================
    xgb = XGBClassifier(
        random_state=1,
        eval_metric="logloss",
        use_label_encoder=False
    )

    param_xgb = {
        "n_estimators": [100, 200],
        "max_depth": [3, 6],
        "learning_rate": [0.01, 0.1],
        "subsample": [0.8, 1.0]
    }

    grid_xgb = GridSearchCV(xgb, param_xgb, cv=cv, scoring="roc_auc", n_jobs=-1)
    grid_xgb.fit(X_train, y_train)

    models["XGBoost"] = grid_xgb

    # =============================
    # COMPARAÇÃO
    # =============================
    results = []

    for name, model in models.items():
        results.append({
            "Model": name,
            "Best Score": model.best_score_,
            "Best Params": model.best_params_,
            "Estimator": model.best_estimator_
        })

    results_df = pd.DataFrame(results).sort_values("Best Score", ascending=False)

    # melhor modelo
    best_model_name = results_df.iloc[0]["Model"]
    best_model = models[best_model_name]

    return {
        "Target": col,
        "Comparison": results_df,
        "Best Model Name": best_model_name,
        "Best Estimator": best_model.best_estimator_,
        "Best Params": best_model.best_params_,
        "Best CV Score": best_model.best_score_,
        "X_train": X_train,
        "Y_train": y_train,
        "X_test": X_test,
        "y_test": y_test
    }

In [108]:
model_results = {}

i = -1
for target_col, deps in target_cols.items():
   # i += 1

   # if i != 3:
   #    continue

   model_results[target_col] = apply_model(target_col, deps)
   print(model_results)

model_results.keys()

c:\Users\Hertz Rafael\Documents\Projetos UFPB\predict-academic-support\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\Hertz Rafael\Documents\Projetos UFPB\predict-academic-support\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\Hertz Rafael\Documents\Projetos UFPB\predict-academic-support\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and

{'target_percntil_media': {'Target': 'target_percntil_media', 'Comparison':           Model  Best Score  \
2       XGBoost    0.955669   
1  RandomForest    0.954517   
0      Logistic    0.938533   

                                         Best Params  \
2  {'learning_rate': 0.1, 'max_depth': 3, 'n_esti...   
1  {'max_depth': 10, 'min_samples_split': 2, 'n_e...   
0          {'model__C': 0.1, 'model__penalty': 'l1'}   

                                           Estimator  
2  XGBClassifier(base_score=None, booster=None, c...  
1  (DecisionTreeClassifier(max_depth=10, max_feat...  
0  (StandardScaler(), LogisticRegression(C=0.1, c...  , 'Best Model Name': 'XGBoost', 'Best Estimator': XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_

c:\Users\Hertz Rafael\Documents\Projetos UFPB\predict-academic-support\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\Hertz Rafael\Documents\Projetos UFPB\predict-academic-support\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\Hertz Rafael\Documents\Projetos UFPB\predict-academic-support\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and

{'target_percntil_media': {'Target': 'target_percntil_media', 'Comparison':           Model  Best Score  \
2       XGBoost    0.955669   
1  RandomForest    0.954517   
0      Logistic    0.938533   

                                         Best Params  \
2  {'learning_rate': 0.1, 'max_depth': 3, 'n_esti...   
1  {'max_depth': 10, 'min_samples_split': 2, 'n_e...   
0          {'model__C': 0.1, 'model__penalty': 'l1'}   

                                           Estimator  
2  XGBClassifier(base_score=None, booster=None, c...  
1  (DecisionTreeClassifier(max_depth=10, max_feat...  
0  (StandardScaler(), LogisticRegression(C=0.1, c...  , 'Best Model Name': 'XGBoost', 'Best Estimator': XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_

c:\Users\Hertz Rafael\Documents\Projetos UFPB\predict-academic-support\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\Hertz Rafael\Documents\Projetos UFPB\predict-academic-support\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\Hertz Rafael\Documents\Projetos UFPB\predict-academic-support\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and

{'target_percntil_media': {'Target': 'target_percntil_media', 'Comparison':           Model  Best Score  \
2       XGBoost    0.955669   
1  RandomForest    0.954517   
0      Logistic    0.938533   

                                         Best Params  \
2  {'learning_rate': 0.1, 'max_depth': 3, 'n_esti...   
1  {'max_depth': 10, 'min_samples_split': 2, 'n_e...   
0          {'model__C': 0.1, 'model__penalty': 'l1'}   

                                           Estimator  
2  XGBClassifier(base_score=None, booster=None, c...  
1  (DecisionTreeClassifier(max_depth=10, max_feat...  
0  (StandardScaler(), LogisticRegression(C=0.1, c...  , 'Best Model Name': 'XGBoost', 'Best Estimator': XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_

c:\Users\Hertz Rafael\Documents\Projetos UFPB\predict-academic-support\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\Hertz Rafael\Documents\Projetos UFPB\predict-academic-support\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\Hertz Rafael\Documents\Projetos UFPB\predict-academic-support\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and

{'target_percntil_media': {'Target': 'target_percntil_media', 'Comparison':           Model  Best Score  \
2       XGBoost    0.955669   
1  RandomForest    0.954517   
0      Logistic    0.938533   

                                         Best Params  \
2  {'learning_rate': 0.1, 'max_depth': 3, 'n_esti...   
1  {'max_depth': 10, 'min_samples_split': 2, 'n_e...   
0          {'model__C': 0.1, 'model__penalty': 'l1'}   

                                           Estimator  
2  XGBClassifier(base_score=None, booster=None, c...  
1  (DecisionTreeClassifier(max_depth=10, max_feat...  
0  (StandardScaler(), LogisticRegression(C=0.1, c...  , 'Best Model Name': 'XGBoost', 'Best Estimator': XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_

dict_keys(['target_percntil_media', 'target_reprov', 'target_trancamento', 'target_risco'])

## target_percentil_media:

### Somente sem media_geral:
{'Target': 'target_percntil_media', 'Best Params': {'model__C': 10, 'model__penalty': 'l1'}, 'Score': np.float64(0.8916015625)}

### Sem media_geral, ch_integralizada e pendente:
{'Target': 'target_percntil_media', 'Best Params': {'model__C': 10, 'model__penalty': 'l1'}, 'Score': np.float64(0.8841796875)}

### Sem media_geral, ch_integralizada, ch_pendente e status_discente:
{'Target': 'target_percntil_media', 'Best Params': {'model__C': 0.01, 'model__penalty': 'l1'}, 'Score': np.float64(0.82138671875)}

### Sem media_geral, ch_integralizada, ch_pendente, status_discente e situacao:
{'Target': 'target_percntil_media', 'Best Params': {'model__C': 10, 'model__penalty': 'l2'}, 'Score': np.float64(0.67666015625)}

### Sem media_geral, status_discente e situacao: <- escolhido
{'Target': 'target_percntil_media', 'Best Params': {'model__C': 0.1, 'model__penalty': 'l1'}, 'Score': np.float64(0.89951171875)}

---

## target_reprov:

### Sem situacao, status_discente: <- escolhido
{'Target': 'target_reprov', 'Best Params': {'model__C': 0.01, 'model__penalty': 'l1'}, 'Score': np.float64(0.79619140625)}

### Sem situacao, status_discente, media_geral:
{'Target': 'target_reprov', 'Best Params': {'model__C': 10, 'model__penalty': 'l1'}, 'Score': np.float64(0.77333984375)}

### Sem situacao, status_discente, media_geral, ch_integralizada e ch_pendente:
{'Target': 'target_reprov', 'Best Params': {'model__C': 0.01, 'model__penalty': 'l1'}, 'Score': np.float64(0.67900390625)}

---

## target_trancamento

### Sem situacao, status_discente: <- escolhido
{'Target': 'target_trancamento', 'Best Params': {'model__C': 10, 'model__penalty': 'l1'}, 'Score': np.float64(0.784765625)}

### Sem situacao, status_discente e media_geral:
{'Target': 'target_trancamento', 'Best Params': {'model__C': 0.1, 'model__penalty': 'l1'}, 'Score': np.float64(0.78671875)}

### Sem situacao, status_discente e faixa_renda_familiar:
{'Target': 'target_trancamento', 'Best Params': {'model__C': 0.1, 'model__penalty': 'l1'}, 'Score': np.float64(0.783203125)}

---

## target_risco

### Sem situacao, status_discente e media_geral:
{'Target': 'target_risco', 'Best Params': {'model__C': 10, 'model__penalty': 'l1'}, 'Score': np.float64(0.9033203125)}

### Sem situacao, status_discente, media_geral, ch_integralizada e ch_pendente:
{'Target': 'target_risco', 'Best Params': {'model__C': 10, 'model__penalty': 'l1'}, 'Score': np.float64(0.7111328125)}

Testar com ambas, avaliar as outras métricas.

In [109]:
import numpy as np

# Calculating Best Threshold
for key, value in model_results.items():
    # prob_train = value['Best Estimator'].predict_proba(value['X_train'])[:, 1]
    # precision, recall, thresholds = precision_recall_curve(value['Y_train'], prob_train)
    # f1_scores = 2 * (precision * recall) / (precision + recall)
    # best_threshold = thresholds[np.argmax(f1_scores)]

    prob_test = value['Best Estimator'].predict_proba(value['X_test'])[:, 1]

    precision, recall, thresholds = precision_recall_curve(value['y_test'], prob_test)
    f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)

    best_threshold = thresholds[np.argmax(f1_scores)]
    
    value['Best Threshold'] = best_threshold

In [116]:
# Avaliação no Teste
for key, value in model_results.items():
    print(key)
    print('Best Model:', value['Best Model Name'])
    prob_test = value['Best Estimator'].predict_proba(value['X_test'])[:, 1]
    pred_test = (prob_test >= value['Best Threshold']).astype(int)
    matriz_confusao = confusion_matrix(value['y_test'], pred_test)
    print(roc_auc_score(value['y_test'], pred_test))
    print(classification_report(value['y_test'], pred_test))
    print('-='*30)

target_percntil_media
Best Model: XGBoost
0.7797467040382021
              precision    recall  f1-score   support

         0.0       0.97      0.97      0.97      2535
         1.0       0.60      0.59      0.59       190

    accuracy                           0.94      2725
   macro avg       0.78      0.78      0.78      2725
weighted avg       0.94      0.94      0.94      2725

-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=
target_reprov
Best Model: XGBoost
0.7198292845303118
              precision    recall  f1-score   support

         0.0       0.92      0.95      0.93      2349
         1.0       0.60      0.49      0.54       376

    accuracy                           0.88      2725
   macro avg       0.76      0.72      0.74      2725
weighted avg       0.88      0.88      0.88      2725

-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=
target_trancamento
Best Model: RandomForest
0.6406629286471331
              precision    recall  f1-score   s

## Resultado com Logit:

target_percntil_media
0.7290356067684003
              precision    recall  f1-score   support

         0.0       0.96      0.93      0.95      2535
         1.0       0.37      0.53      0.43       190

    accuracy                           0.90      2725
   macro avg       0.66      0.73      0.69      2725
weighted avg       0.92      0.90      0.91      2725

-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=
target_reprov
0.7054190103529795
              precision    recall  f1-score   support

         0.0       0.92      0.97      0.94      2349
         1.0       0.70      0.44      0.54       376

    accuracy                           0.90      2725
   macro avg       0.81      0.71      0.74      2725
weighted avg       0.89      0.90      0.89      2725

-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=
target_trancamento
0.6176393494934299
              precision    recall  f1-score   support

         0.0       0.98      0.99      0.99      2659
         1.0       0.46      0.24      0.32        66

    accuracy                           0.97      2725
   macro avg       0.72      0.62      0.65      2725
weighted avg       0.97      0.97      0.97      2725

-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=
target_risco
0.5999357663468092
              precision    recall  f1-score   support

         0.0       0.97      0.99      0.98      2608
         1.0       0.41      0.21      0.28       117

    accuracy                           0.95      2725
   macro avg       0.69      0.60      0.63      2725
weighted avg       0.94      0.95      0.95      2725

-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=-=

# Resultado:

Melhor Target: target_percentil_media
Modelo: XGBoost
Métricas:
              precision    recall  f1-score   support

         0.0       0.97      0.97      0.97      2535
         1.0       0.60      0.59      0.59       190

    accuracy                           0.94      2725
   macro avg       0.78      0.78      0.78      2725
weighted avg       0.94      0.94      0.94      2725